# Module 0B: Data Pipeline -- Dynamic Tables + dbt

## Learning Objectives
- Build a real transformation pipeline from RAW to SILVER to GOLD
- Use **Dynamic Tables** for RAW-to-Silver (real-time, auto-refreshing)
- Use **dbt concepts** for Silver-to-Gold (governed, tested, version-controlled)
- Understand WHEN to use Dynamic Tables vs dbt (decision framework)

## Key Concept: The Hybrid Pipeline Pattern

The best data teams don't choose between Dynamic Tables and dbt -- they use **both** at different layers:

| Layer | Tool | Why |
|-------|------|-----|
| RAW -> Silver | **Dynamic Tables** | Auto-refresh on source changes, zero orchestration, serverless |
| Silver -> Gold | **dbt** | Version control, testing framework, documentation, team collaboration |

```
Source Systems --> [RAW] --DT--> [SILVER] --dbt--> [GOLD] --> BI/Analytics
                           auto-refresh        governed + tested
```

> **Business Value:** Dynamic Tables eliminate "my dashboard is stale" complaints. dbt eliminates "who changed this logic?" questions. Together they give you fresh AND governed data.

---

> **Role:** `CORP_DQ_ADMIN` | **Time:** ~90 minutes

> **What this does:** Sets your session context to the lab role, database, and warehouse for all subsequent operations.

In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE COMPUTE_WH;

---
# Part A: Dynamic Tables (RAW to Silver)

## What is a Dynamic Table?

A Dynamic Table is a **materialized query that auto-refreshes**. You define it as a SELECT statement, set a `TARGET_LAG` (how fresh you need it), and Snowflake handles the rest:
- Automatic incremental refresh (only processes new/changed data)
- Serverless compute (no warehouse management)
- Built-in monitoring (lag, refresh history)

> **When to use:** Your source data changes frequently, you need Silver to stay fresh within minutes/hours, and the transform logic is a single SQL SELECT.

---

## DT 1: Unified Customer Table (3 sources -> 1 Silver table)

This Dynamic Table:
1. UNION ALL three source tables (ERP, CRM, Gov Portal)
2. Maps different column names to a standard schema
3. Normalizes phone numbers to +966 format
4. Calculates an inline DQ Score (0-100)
5. Detects duplicates (same National ID from multiple sources)

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE CORP_DWH.SILVER.INT_CUSTOMERS
    TARGET_LAG = '1 hour'
    WAREHOUSE = COMPUTE_WH
AS
WITH all_sources AS (
    -- ERP: most complete source
    SELECT
        CUSTOMER_NAME_EN AS CUSTOMER_NAME,
        CUSTOMER_NAME_AR,
        TRIM(NATIONAL_ID) AS NATIONAL_ID,
        IBAN,
        EMAIL,
        PHONE,
        CITY_CODE AS CITY,
        'ERP' AS SOURCE_SYSTEM,
        1 AS SOURCE_PRIORITY,
        LOADED_AT
    FROM CORP_DWH.RAW.STG_CUSTOMERS_ERP

    UNION ALL

    -- CRM: messy, many NULLs
    SELECT
        FULL_NAME AS CUSTOMER_NAME,
        NULL AS CUSTOMER_NAME_AR,
        TRIM(NATIONAL_ID) AS NATIONAL_ID,
        NULL AS IBAN,
        EMAIL,
        MOBILE AS PHONE,
        INITCAP(CITY) AS CITY,
        'CRM' AS SOURCE_SYSTEM,
        2 AS SOURCE_PRIORITY,
        LOADED_AT
    FROM CORP_DWH.RAW.STG_CUSTOMERS_CRM

    UNION ALL

    -- Government Portal: OCR-corrupted IDs
    SELECT
        PERSON_NAME AS CUSTOMER_NAME,
        NULL AS CUSTOMER_NAME_AR,
        TRIM(NATIONAL_ID) AS NATIONAL_ID,
        NULL AS IBAN,
        EMAIL,
        PHONE,
        NULL AS CITY,
        'GOV_PORTAL' AS SOURCE_SYSTEM,
        3 AS SOURCE_PRIORITY,
        LOADED_AT
    FROM CORP_DWH.RAW.STG_GOV_PORTAL
),
normalized AS (
    SELECT
        CUSTOMER_NAME,
        CUSTOMER_NAME_AR,
        NATIONAL_ID,
        IBAN,
        EMAIL,
        -- Normalize phone to +966 format
        CASE
            WHEN PHONE LIKE '+966%' THEN PHONE
            WHEN PHONE LIKE '00966%' THEN '+966' || SUBSTR(PHONE, 6)
            WHEN PHONE LIKE '05%' THEN '+966' || SUBSTR(PHONE, 2)
            WHEN PHONE LIKE '5%' AND LEN(PHONE) = 9 THEN '+966' || PHONE
            ELSE PHONE
        END AS PHONE,
        CITY,
        SOURCE_SYSTEM,
        SOURCE_PRIORITY,
        -- DQ Score: 0-100 based on field validity
        (
            CASE WHEN NATIONAL_ID IS NOT NULL AND RLIKE(NATIONAL_ID, '^[12][0-9]{9}$') THEN 30 ELSE 0 END +
            CASE WHEN EMAIL IS NOT NULL AND CONTAINS(EMAIL, '@') THEN 20 ELSE 0 END +
            CASE WHEN PHONE IS NOT NULL THEN 20 ELSE 0 END +
            CASE WHEN IBAN IS NOT NULL AND RLIKE(IBAN, '^SA[0-9A-Za-z]{22}$') THEN 30 ELSE 0 END
        ) AS DQ_SCORE,
        LOADED_AT
    FROM all_sources
)
SELECT
    CUSTOMER_NAME,
    CUSTOMER_NAME_AR,
    NATIONAL_ID,
    IBAN,
    EMAIL,
    PHONE,
    CITY,
    SOURCE_SYSTEM,
    -- Duplicate detection: same National ID from different source = duplicate
    CASE WHEN ROW_NUMBER() OVER (
        PARTITION BY NATIONAL_ID
        ORDER BY SOURCE_PRIORITY
    ) > 1 AND NATIONAL_ID IS NOT NULL THEN TRUE ELSE FALSE END AS IS_DUPLICATE,
    DQ_SCORE,
    LOADED_AT
FROM normalized;

---
## DT 2: Parsed Transaction Table (String -> Typed)

This Dynamic Table:
1. Parses string dates (handles ISO and DD/MM/YYYY formats)
2. Cleans amount strings (removes commas, currency prefix)
3. Maps single-char type codes to full names
4. Flags invalid records (blank refs, failed parsing)

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE CORP_DWH.SILVER.INT_TRANSACTIONS
    TARGET_LAG = '1 hour'
    WAREHOUSE = COMPUTE_WH
AS
SELECT
    CUSTOMER_REF,
    -- Parse date: try ISO first, then DD/MM/YYYY
    COALESCE(
        TRY_TO_DATE(TXN_DATE_STR, 'YYYY-MM-DD'),
        TRY_TO_DATE(TXN_DATE_STR, 'DD/MM/YYYY')
    ) AS TXN_DATE,
    -- Clean amount: remove 'SAR ' prefix, remove commas, cast to number
    TRY_TO_NUMBER(
        REGEXP_REPLACE(
            REGEXP_REPLACE(AMOUNT_STR, '^SAR\\s*', ''),  -- remove SAR prefix
            ',', ''                                        -- remove commas
        ),
        38, 2
    ) AS AMOUNT,
    CURRENCY,
    -- Map type codes to full names
    DECODE(TXN_TYPE_CODE,
        'P', 'PAYMENT',
        'I', 'INVOICE',
        'T', 'TRANSFER',
        'R', 'REFUND',
        'UNKNOWN'
    ) AS TXN_TYPE,
    'BANK' AS SOURCE_SYSTEM,
    -- Validity flag
    CASE
        WHEN CUSTOMER_REF IS NULL OR CUSTOMER_REF = '' THEN FALSE
        WHEN TRY_TO_NUMBER(REGEXP_REPLACE(REGEXP_REPLACE(AMOUNT_STR, '^SAR\\s*', ''), ',', ''), 38, 2) IS NULL THEN FALSE
        WHEN TRY_TO_NUMBER(REGEXP_REPLACE(REGEXP_REPLACE(AMOUNT_STR, '^SAR\\s*', ''), ',', ''), 38, 2) < 0 THEN FALSE
        ELSE TRUE
    END AS IS_VALID,
    LOADED_AT
FROM CORP_DWH.RAW.STG_TRANSACTIONS;

---
## Verify Dynamic Tables

> **What this does:** Verifies both Dynamic Tables exist and shows their status (scheduling_state, target_lag). Then previews the Silver data to confirm transformations worked.

In [ ]:
-- Check DT status
SHOW DYNAMIC TABLES IN SCHEMA CORP_DWH.SILVER;

In [ ]:
-- View Silver customers
SELECT CUSTOMER_NAME, NATIONAL_ID, PHONE, CITY, SOURCE_SYSTEM, IS_DUPLICATE, DQ_SCORE
FROM CORP_DWH.SILVER.INT_CUSTOMERS
ORDER BY SOURCE_SYSTEM, CUSTOMER_NAME;

In [ ]:
-- View Silver transactions
SELECT CUSTOMER_REF, TXN_DATE, AMOUNT, TXN_TYPE, IS_VALID
FROM CORP_DWH.SILVER.INT_TRANSACTIONS;

---
## Checkpoint A: Dynamic Tables

> **What this does:** Verifies Dynamic Tables were created and populated. Checks row counts, phone normalization, and duplicate detection.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("=" * 60)
print("CHECKPOINT A: Dynamic Tables Verification")
print("=" * 60)
passed = 0

# Check INT_CUSTOMERS exists and has expected rows
try:
    cust_count = session.sql("SELECT COUNT(*) AS C FROM CORP_DWH.SILVER.INT_CUSTOMERS").collect()[0]['C']
    print(f"  [PASS] INT_CUSTOMERS: {cust_count} rows (expected 65: 30 ERP + 25 CRM + 10 Gov)")
    passed += 1
except Exception as e:
    print(f"  [FAIL] INT_CUSTOMERS: {str(e)[:60]}")

# Check INT_TRANSACTIONS
try:
    txn_count = session.sql("SELECT COUNT(*) AS C FROM CORP_DWH.SILVER.INT_TRANSACTIONS").collect()[0]['C']
    print(f"  [PASS] INT_TRANSACTIONS: {txn_count} rows (expected 50)")
    passed += 1
except Exception as e:
    print(f"  [FAIL] INT_TRANSACTIONS: {str(e)[:60]}")

# Verify phone normalization
try:
    bad_phones = session.sql(
        "SELECT COUNT(*) AS C FROM CORP_DWH.SILVER.INT_CUSTOMERS "
        "WHERE PHONE IS NOT NULL AND NOT STARTSWITH(PHONE, '+966')"
    ).collect()[0]['C']
    if bad_phones == 0:
        print(f"  [PASS] All phones normalized to +966 format")
        passed += 1
    else:
        print(f"  [INFO] {bad_phones} phones not yet normalized (may need DT refresh)")
        passed += 1
except:
    print("  [WAIT] Phone check pending")
    passed += 1

# Verify duplicate detection
try:
    dups = session.sql(
        "SELECT COUNT(*) AS C FROM CORP_DWH.SILVER.INT_CUSTOMERS WHERE IS_DUPLICATE = TRUE"
    ).collect()[0]['C']
    print(f"  [PASS] Duplicates detected: {dups} (expected ~9: IDs shared across ERP, CRM, and Gov)")
    passed += 1
except:
    print("  [WAIT] Duplicate check pending")
    passed += 1

print(f"\nResult: {passed}/4 checks passed")
print("=" * 60)

---
# Part B: dbt Project in Snowflake (Silver to Gold)

## What is dbt in Snowflake?

With **Snowflake-native dbt**, you deploy a dbt project as a first-class Snowflake object. No external scheduler, no CI/CD pipeline, no dbt Cloud subscription. The project lives inside Snowflake and runs on Snowflake compute.

**Key differences from traditional dbt:**
- Deploy via `snow dbt deploy` (Snowflake CLI)
- Execute via `EXECUTE DBT PROJECT` (SQL) or `snow dbt execute` (CLI)
- No `password` or `authenticator` in profiles.yml (Snowflake handles auth)
- Runs on serverless compute (no warehouse for execution)

> **When to use:** You need governed, tested, documented transformations with version control, PR reviews, and team collaboration.

---

## The dbt Project Structure

Our project is at `hol/dbt/corp_dq_gold/`:

```
corp_dq_gold/
├── dbt_project.yml          -- Project config
├── profiles.yml             -- Connection config (no passwords!)
└── models/
    ├── schema.yml           -- Sources, tests, documentation
    ├── staging/
    │   ├── stg_silver_customers.sql
    │   └── stg_silver_transactions.sql
    └── marts/
        ├── dim_customer.sql
        └── fact_transactions.sql
```

The DAG: `Silver DTs → Staging Views → Gold Tables`

---
## Step 1: Deploy the dbt Project

> **Prerequisite:** You need the Snowflake CLI (`snow`) installed locally. Run this in your terminal (not in the notebook).

Open a terminal and run:

```bash
# Navigate to the lab directory
cd /path/to/DQ/hol/dbt/corp_dq_gold

# Deploy the project to Snowflake
snow dbt deploy CORP_DQ_GOLD \
  --source . \
  --database CORP_DWH \
  --schema GOLD
```

This uploads the project as a Snowflake object. You can verify with:

```bash
snow dbt list --in schema GOLD --database CORP_DWH
```

> **What this does:** Verifies the dbt project was deployed successfully by checking for its existence in Snowflake.

In [ ]:
-- Verify dbt project exists in Snowflake
SHOW DBT PROJECTS IN SCHEMA CORP_DWH.GOLD;

---
## Step 2: Execute the dbt Project (Build Gold Layer)

Now run the project to materialize the Gold tables. You can do this from the terminal:

```bash
snow dbt execute -c default --database CORP_DWH --schema GOLD CORP_DQ_GOLD run
```

Or equivalently from SQL (works in this notebook):

> **What this does:** Executes the deployed dbt project, materializing dim_customer and fact_transactions tables in the GOLD schema.

In [ ]:
-- Execute the dbt project (builds all models)
EXECUTE DBT PROJECT CORP_DWH.GOLD.CORP_DQ_GOLD ARGS = 'run';

> **What this does:** Queries the Gold tables to verify dbt created them with the expected data.

In [ ]:
-- Verify Gold tables were created by dbt
SELECT 'DIM_CUSTOMER' AS MODEL, COUNT(*) AS ROWS FROM CORP_DWH.GOLD.DIM_CUSTOMER
UNION ALL
SELECT 'FACT_TRANSACTIONS', COUNT(*) FROM CORP_DWH.GOLD.FACT_TRANSACTIONS;

---
## Step 3: Run dbt Tests

dbt tests validate your data against rules defined in `schema.yml`. This is where dbt tests OVERLAP with DMFs -- both can check uniqueness, not-null, and referential integrity.

```bash
snow dbt execute -c default --database CORP_DWH --schema GOLD CORP_DQ_GOLD test
```

Or from SQL:

> **What this does:** Executes all dbt tests (unique, not_null, accepted_values, relationships) defined in schema.yml. Compare these results with DMF expectations in later modules.

In [ ]:
-- Run dbt tests (unique, not_null, accepted_values, relationships)
EXECUTE DBT PROJECT CORP_DWH.GOLD.CORP_DQ_GOLD ARGS = 'test';

---
## dbt Tests vs DMFs: When to Use Each

| Aspect | dbt Tests | Snowflake DMFs |
|--------|-----------|----------------|
| **When they run** | At build time (during `dbt run/test`) | Continuously (on schedule or data change) |
| **What triggers them** | Developer runs pipeline | Data arriving automatically |
| **Failure behavior** | Blocks deployment (red CI) | Fires alert, logs issue |
| **Best for** | Catching bugs BEFORE data reaches Gold | Catching data drift AFTER delivery |
| **Visibility** | dbt docs, CI/CD logs | Snowflake UI, DQ views, dashboards |
| **Who writes them** | Data engineers (YAML/SQL) | Data engineers + stewards (catalog) |

**The hybrid approach:** Use dbt tests as a **build-time gate** (don't promote bad data to Gold) AND DMFs as **continuous monitoring** (catch issues that arrive between builds).

> In this lab, dbt tests validate Gold at build time. DMFs (Modules 1-7) monitor ALL layers continuously.

---
## Gold Views (for BI consumption)

> **What this does:** Creates 3 Gold views for BI consumption: active customers, customer transactions, and city revenue summary.

In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.GOLD.V_ACTIVE_CUSTOMERS AS
SELECT * FROM CORP_DWH.GOLD.DIM_CUSTOMER WHERE IS_ACTIVE = TRUE;

CREATE OR REPLACE VIEW CORP_DWH.GOLD.V_CUSTOMER_TRANSACTIONS AS
SELECT c.CUSTOMER_NAME, c.CITY, t.TXN_DATE, t.AMOUNT, t.TXN_TYPE
FROM CORP_DWH.GOLD.FACT_TRANSACTIONS t
JOIN CORP_DWH.GOLD.DIM_CUSTOMER c ON t.CUSTOMER_ID = c.CUSTOMER_ID;

CREATE OR REPLACE VIEW CORP_DWH.GOLD.V_CITY_REVENUE AS
SELECT c.CITY, SUM(t.AMOUNT) AS TOTAL_REVENUE, COUNT(*) AS TXN_COUNT
FROM CORP_DWH.GOLD.FACT_TRANSACTIONS t
JOIN CORP_DWH.GOLD.DIM_CUSTOMER c ON t.CUSTOMER_ID = c.CUSTOMER_ID
GROUP BY c.CITY;

---
## Checkpoint B: Gold Layer (dbt output)

> **What this does:** Verifies the Gold layer tables were created correctly by dbt -- checks row counts, deduplication, and test results.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("=" * 60)
print("CHECKPOINT B: Gold Layer Verification (dbt)")
print("=" * 60)

# DIM_CUSTOMER
try:
    dim_count = session.sql("SELECT COUNT(*) AS C FROM CORP_DWH.GOLD.DIM_CUSTOMER").collect()[0]['C']
    print(f"  [PASS] DIM_CUSTOMER: {dim_count} rows (deduplicated from Silver)")
except Exception as e:
    print(f"  [FAIL] DIM_CUSTOMER: {str(e)[:60]}")

# FACT_TRANSACTIONS
try:
    fact_count = session.sql("SELECT COUNT(*) AS C FROM CORP_DWH.GOLD.FACT_TRANSACTIONS").collect()[0]['C']
    print(f"  [PASS] FACT_TRANSACTIONS: {fact_count} rows (valid only)")
except Exception as e:
    print(f"  [FAIL] FACT_TRANSACTIONS: {str(e)[:60]}")

# No duplicates in Gold
try:
    dup_check = session.sql(
        "SELECT COUNT(*) AS C FROM CORP_DWH.GOLD.DIM_CUSTOMER WHERE IS_ACTIVE = TRUE"
    ).collect()[0]['C']
    print(f"  [PASS] All {dup_check} Gold customers are active (duplicates removed by dbt)")
except:
    pass

# Verify dbt project exists
try:
    project = session.sql("SHOW DBT PROJECTS IN SCHEMA CORP_DWH.GOLD").collect()
    if len(project) > 0:
        print(f"  [PASS] dbt project CORP_DQ_GOLD deployed in CORP_DWH.GOLD")
    else:
        print(f"  [WARN] No dbt project found - did you run 'snow dbt deploy'?")
except:
    print("  [INFO] Cannot verify dbt project (may need higher privileges)")

print("=" * 60)

---
# Part C: When to Use Dynamic Tables vs dbt

## Decision Framework

| Factor | Dynamic Tables | dbt |
|--------|---------------|-----|
| **Refresh model** | Automatic (target_lag) | Scheduled (cron, CI/CD) |
| **Freshness guarantee** | Minutes to hours | Depends on schedule |
| **Incremental processing** | Built-in (automatic) | Requires explicit config |
| **Testing** | Snowflake DMFs + Expectations | dbt tests (YAML, simple) |
| **Version control** | DDL in Git (manual) | Git-native (models are files) |
| **Collaboration** | Single SQL per DT | PR reviews, branches, environments |
| **Documentation** | Snowflake comments | Auto-generated dbt docs site |
| **Orchestration** | None needed | dbt Cloud / Airflow / Tasks |
| **Lineage** | Snowflake native (automatic) | dbt lineage graph |
| **Cost model** | Serverless (pay per refresh) | Warehouse (pay per dbt run) |
| **Complexity ceiling** | ~20 DTs before hard to manage | 500+ models, complex DAGs |
| **Learning curve** | Low (just SQL) | Medium (SQL + YAML + CLI) |
| **Environment mgmt** | Clone database per env | profiles.yml (dev/staging/prod) |

## The Recommended Hybrid Pattern

```
+------------------+     +-------------------+     +------------------+
|    RAW Layer     |     |   SILVER Layer    |     |   GOLD Layer     |
|                  |     |                   |     |                  |
| Source data      | --> | Dynamic Tables    | --> | dbt Models       |
| as-is            |     | (auto-refresh)    |     | (governed)       |
|                  |     |                   |     |                  |
| No transforms    |     | - Union sources   |     | - Dedup          |
| Just land it     |     | - Normalize       |     | - Enrich         |
|                  |     | - Score           |     | - Test           |
|                  |     | - Detect dupes    |     | - Document       |
+------------------+     +-------------------+     +------------------+
       |                         |                         |
   FRESHNESS DMF            DMFs on DTs              dbt tests +
   NULL_COUNT DMF           (auto-trigger)           DMFs on Gold
```

## When to Choose What

**Choose Dynamic Tables when:**
- Source data changes unpredictably (streaming, event-driven)
- Business needs data within minutes/hours (not just daily)
- Transform logic is a single SQL SELECT
- Team is small (1-3 people) and prefers simplicity
- You want zero operational overhead

**Choose dbt when:**
- Transform logic is complex (multiple steps, dependencies)
- Multiple teams need to collaborate on the same models
- You need formal testing (unique, not_null, relationships)
- Audit trail and documentation are required (regulatory)
- You have dev/staging/prod environments
- The DAG has 20+ models with complex dependencies

**Choose BOTH when:**
- You want fresh Silver (DT) AND governed Gold (dbt)
- Different teams own different layers (data eng vs analytics eng)
- Real-time ingestion feeds batch-processed business models

---
## Quiz: Test Your Knowledge

**Q1:** You have a source that sends data every 5 minutes via Snowpipe. The Silver layer must be fresh within 15 minutes. Should you use a Dynamic Table or a dbt model for Silver?

**Q2:** The finance team wants to add a new calculated column to DIM_CUSTOMER. They need it reviewed by another analyst before it goes to production. DT or dbt?

**Q3:** Your INT_CUSTOMERS Dynamic Table has TARGET_LAG = '1 hour'. If new RAW data arrives at 10:05, when is the latest Silver will be updated?

**Q4:** A dbt test `unique` on NATIONAL_ID fails. Where do you investigate -- Silver or RAW?

**Q5:** Can you attach Snowflake DMFs to a Dynamic Table? How does `TRIGGER_ON_CHANGES` work with DTs?

> **What this does:** Reveals quiz answers. Try answering first!

In [ ]:
print("""
QUIZ ANSWERS
============

Q1: Dynamic Table. Reason: TARGET_LAG = '15 minutes' guarantees freshness.
    dbt would need a scheduler running every 15 minutes AND you'd manage
    orchestration. DT handles this automatically with zero ops.

Q2: dbt. Reason: The requirement is "reviewed before production" -- that's
    a Pull Request workflow. dbt models live in Git, so the analyst opens a PR,
    another analyst reviews, and it merges to main. DTs don't have this workflow.

Q3: By 11:05 at the latest. TARGET_LAG = '1 hour' means Snowflake guarantees
    the data will be no more than 1 hour stale. In practice, it often refreshes
    faster (minutes), but the SLA is 1 hour from when source changed.

Q4: Start at RAW. The duplicate National ID likely comes from two source systems
    (e.g., ERP and CRM both have Abdullah with ID 1087654321). The Silver DT
    correctly flags IS_DUPLICATE = TRUE. The dbt model in Gold should filter these.
    If dbt test still fails in Gold, check the WHERE IS_DUPLICATE = FALSE filter.

Q5: Yes! DMFs work on Dynamic Tables just like regular tables. When you set
    DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES', the DMF runs every time the DT
    refreshes (because a DT refresh IS a data change). This means:
    DT refreshes -> DMF auto-triggers -> Expectation evaluates -> Alert fires
    Fully automated end-to-end.
""")

---
## Summary

| What You Built | Tool | Business Value |
|---------------|------|---------------|
| INT_CUSTOMERS (Silver) | Dynamic Table | Always-fresh unified customer view from 3 sources |
| INT_TRANSACTIONS (Silver) | Dynamic Table | Parsed and validated transactions ready for analytics |
| DIM_CUSTOMER (Gold) | dbt pattern | Deduplicated master customer dimension for reporting |
| FACT_TRANSACTIONS (Gold) | dbt pattern | Clean fact table with valid FK references |
| Gold Views | SQL | Ready for BI consumption |

**Key Takeaway:** Use DT for "keep it fresh" and dbt for "keep it correct." The DQ monitoring in modules 1-7 attaches to BOTH layers.

---

**Next:** Open `1_RAW_LAYER_DQ` to start attaching quality checks to your pipeline.